In [1]:
import pandas as pd
from pathlib import Path

# Ruta del archivo limpio
file_path = r"C:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\data\interim\meteo_data_with_generation_clean\parque_solar_girasol_clean.parquet"

# Cargar el DataFrame
df = pd.read_parquet(file_path)
print("Forma del DataFrame limpio:", df.shape)

# Asegurarse de que la columna de tiempo se llame 'timestamp'
if 'timestamp' not in df.columns and 'date' in df.columns:
    df = df.rename(columns={"date": "timestamp"})

# Convertir 'timestamp' a datetime (si no lo está)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Separar el target 'generation' del resto de features
if "generation" not in df.columns:
    raise KeyError("No se encontró la columna 'generation' en el DataFrame.")

X = df.drop(columns=["generation"])
y = df["generation"]

print("Número de features:", X.shape[1])
print("Número de registros para target:", y.shape[0])

# Confirma la ejecución y dime si continuamos con la siguiente celda.


Forma del DataFrame limpio: (25495, 241)
Número de features: 240
Número de registros para target: 25495


In [4]:
print(X.columns.tolist())

['timestamp', 'temperature_2m', 'dew_point_2m', 'relative_humidity_2m', 'apparent_temperature', 'surface_pressure', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'is_day', 'sunshine_duration', 'wet_bulb_temperature_2m', 'boundary_layer_height', 'shortwave_radiation', 'diffuse_radiation', 'global_tilted_irradiance', 'shortwave_radiation_instant', 'diffuse_radiation_instant', 'global_tilted_irradiance_instant', 'direct_radiation', 'direct_normal_irradiance', 'terrestrial_radiation', 'direct_radiation_instant', 'direct_normal_irradiance_instant', 'terrestrial_radiation_instant', 'pressure_msl', 'hour', 'hour_sin', 'hour_cos', 'day_of_week', 'dow_sin', 'dow_cos', 'month', 'month_sin', 'month_cos', 'generation_lag1', 'generation_lag2', 'generation_lag3', 'temperature_2m_lag1', 'temperature_2m_lag2', 'temperature_2m_lag3', 'dew_point_2m_lag1', 'dew_point_2m_lag2', 'dew_po

In [2]:
from sklearn.model_selection import train_test_split

# Para series temporales, es recomendable no barajar; usaremos slicing basado en el orden.
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test  = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test  = y.iloc[split_index:].copy()

print("Tamaño del set de entrenamiento:", X_train.shape[0])
print("Tamaño del set de prueba:", X_test.shape[0])

Tamaño del set de entrenamiento: 20396
Tamaño del set de prueba: 5099


In [3]:
from sklearn.preprocessing import StandardScaler

# Eliminar la columna "timestamp" de X_train y X_test, ya que no es numérica
X_train_features = X_train.drop(columns=["timestamp"], errors="ignore")
X_test_features  = X_test.drop(columns=["timestamp"], errors="ignore")

# Inicializar el escalador
scaler = StandardScaler()

# Ajustar el escalador en el set de entrenamiento y transformar ambos conjuntos
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_features), 
                              columns=X_train_features.columns, 
                              index=X_train_features.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_features), 
                             columns=X_test_features.columns, 
                             index=X_test_features.index)

print("Primeras 5 filas del set de entrenamiento escalado:")
X_train_scaled.head()

Primeras 5 filas del set de entrenamiento escalado:


,temperature_2m,dew_point_2m,relative_humidity_2m,apparent_temperature,surface_pressure,cloud_cover,cloud_cover_low,cloud_cover_mid,et0_fao_evapotranspiration,vapour_pressure_deficit,...,dow_sin_ma3,dow_sin_ma6,dow_cos_ma3,dow_cos_ma6,month_ma3,month_ma6,month_sin_ma3,month_sin_ma6,month_cos_ma3,month_cos_ma6
0,0.190615,0.813479,0.411881,0.200620,0.039089,-0.960431,-0.280205,-0.366994,-0.689386,-0.368464,...,0.370324,0.279711,1.251995,1.307566,0.545582,0.545832,-1.307764,-1.307702,-0.114324,-0.114237
1,0.074953,0.721057,0.449019,0.090810,-0.092747,-0.799961,-0.680538,-0.641872,-0.699501,-0.419588,...,0.741165,0.448017,1.073101,1.226383,0.545582,0.545832,-1.307764,-1.307702,-0.114324,-0.114237
2,-0.002156,0.536214,0.365844,-0.029273,-0.393099,-0.666236,-0.380288,-0.421970,-0.689063,-0.371998,...,1.112007,0.560221,0.894206,1.172260,0.545582,0.545832,-1.307764,-1.307702,-0.114324,-0.114237
3,-0.002156,0.420687,0.265860,-0.012057,-0.563108,-0.586001,0.020045,0.512617,-0.694302,-0.297027,...,1.112007,0.747227,0.894206,1.082056,0.545582,0.545832,-1.307764,-1.307702,-0.114324,-0.114237
4,-0.079264,0.536214,0.444571,-0.033935,-0.565937,-0.612746,0.070087,0.567592,-0.718345,-0.445688,...,1.112007,0.934233,0.894206,0.991852,0.545582,0.545832,-1.307764,-1.307702,-0.114324,-0.114237


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import joblib
from pathlib import Path

def grid_search_model(model, param_grid, X_train, y_train, cv=3, n_jobs=2):
    grid = GridSearchCV(model, param_grid, cv=cv, scoring="r2", n_jobs=n_jobs)
    grid.fit(X_train, y_train)
    print(f"Mejores hiperparámetros para {model.__class__.__name__}: {grid.best_params_}")
    return grid.best_estimator_

print("Entrenando modelos base (con n_jobs=2)...")
models = {
    "RF": grid_search_model(RandomForestRegressor(random_state=42), 
                            {"n_estimators": [50, 100], "max_depth": [5, 10]}, 
                            X_train_scaled, y_train, n_jobs=2),
    "GB": grid_search_model(GradientBoostingRegressor(random_state=42), 
                            {"n_estimators": [50, 100], "learning_rate": [0.01, 0.1], "max_depth": [3, 5]}, 
                            X_train_scaled, y_train, n_jobs=2),
    "XGB": grid_search_model(XGBRegressor(objective="reg:squarederror", random_state=42), 
                             {"n_estimators": [50, 100], "max_depth": [3, 5], "learning_rate": [0.01, 0.1]}, 
                             X_train_scaled, y_train, n_jobs=2),
    "LGB": grid_search_model(LGBMRegressor(random_state=42), 
                             {"n_estimators": [50, 100], "max_depth": [3, 5], "learning_rate": [0.01, 0.1]}, 
                             X_train_scaled, y_train, n_jobs=2),
    "SVR": grid_search_model(SVR(), 
                             {"C": [0.1, 1, 10], "epsilon": [0.01, 0.1]}, 
                             X_train_scaled, y_train, n_jobs=2)
}

print("Entrenando el ensemble de stacking (con n_jobs=2)...")
ensemble = StackingRegressor(estimators=list(models.items()), final_estimator=Ridge(), cv=3, n_jobs=2)
ensemble.fit(X_train_scaled, y_train)

y_pred = ensemble.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"\nStacking Ensemble R2: {r2:.4f}")
print(f"Stacking Ensemble MSE: {mse:.4f}")

# Guardar el modelo ensemble en la ruta especificada
output_folder = Path(r"C:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\data\models")
output_folder.mkdir(parents=True, exist_ok=True)
model_filename = output_folder / "parque_solar_girasol_ensemble.joblib"
joblib.dump(ensemble, model_filename)
print(f"Modelo ensemble guardado en: {model_filename}")

Entrenando modelos base (con n_jobs=2)...
Mejores hiperparámetros para RandomForestRegressor: {'max_depth': 10, 'n_estimators': 100}
Mejores hiperparámetros para GradientBoostingRegressor: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
Mejores hiperparámetros para XGBRegressor: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}


c:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\my_environment\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ferna\Documents\Desktop\11 - Masters\00 - Master AI\99 - Proyecto Final\energy-generation-prediction-dashboard\my_environment\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\ferna\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Us

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016568 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43791
[LightGBM] [Info] Number of data points in the train set: 20396, number of used features: 239
[LightGBM] [Info] Start training from score 25.523316
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

: 